In [2]:
!pip install langgraph


[notice] A new release of pip available: 22.3 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict
import time

In [4]:
# 1. Define the state
class CrashState(TypedDict):
    input: str
    step1: str
    step2: str

In [5]:
# 2. Define steps
def step_1(state: CrashState) -> CrashState:
    print("✅ Step 1 executed")
    return {"step1": "done", "input": state["input"]}

def step_2(state: CrashState) -> CrashState:
    print("⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)")
    time.sleep(60)  # Simulate long-running hang
    return {"step2": "done"}

def step_3(state: CrashState) -> CrashState:
    print("✅ Step 3 executed")
    return {"done": True}

In [6]:
# 3. Build the graph
builder = StateGraph(CrashState)
builder.add_node("step_1", step_1)
builder.add_node("step_2", step_2)
builder.add_node("step_3", step_3)

builder.set_entry_point("step_1")
builder.add_edge("step_1", "step_2")
builder.add_edge("step_2", "step_3")
builder.add_edge("step_3", END)

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [7]:
try:
    print("▶️ Running graph: Please manually interrupt during Step 2...")
    config = {"configurable": {"thread_id": 'thread-1'}}
    graph.invoke({"input": "start"}, config=config)
except KeyboardInterrupt:
    print("❌ Kernel manually interrupted (crash simulated).")

▶️ Running graph: Please manually interrupt during Step 2...
✅ Step 1 executed
⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)
✅ Step 3 executed


In [8]:
graph.get_state(config)

StateSnapshot(values={'input': 'start', 'step1': 'done', 'step2': 'done'}, next=(), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f0eb103-4977-655c-8003-960bf33c5dea'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-01-06T14:58:48.283068+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f0eb103-4973-6279-8002-e1de8f6fc5b7'}}, tasks=(), interrupts=())

In [9]:
list(graph.get_state_history(config))

[StateSnapshot(values={'input': 'start', 'step1': 'done', 'step2': 'done'}, next=(), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f0eb103-4977-655c-8003-960bf33c5dea'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-01-06T14:58:48.283068+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f0eb103-4973-6279-8002-e1de8f6fc5b7'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'input': 'start', 'step1': 'done', 'step2': 'done'}, next=('step_3',), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f0eb103-4973-6279-8002-e1de8f6fc5b7'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-01-06T14:58:48.281356+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f0eb101-0d38-6168-8001-39b6477fe025'}}, tasks=(PregelTask(id='1290ba85-3dea-ad22-be70-1195efd97d1

In [10]:
graph.invoke(None, config=config)

{'input': 'start', 'step1': 'done', 'step2': 'done'}

In [11]:
graph.get_state(config)

StateSnapshot(values={'input': 'start', 'step1': 'done', 'step2': 'done'}, next=(), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f0eb103-4977-655c-8003-960bf33c5dea'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-01-06T14:58:48.283068+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f0eb103-4973-6279-8002-e1de8f6fc5b7'}}, tasks=(), interrupts=())

In [12]:
list(graph.get_state_history(config))

[StateSnapshot(values={'input': 'start', 'step1': 'done', 'step2': 'done'}, next=(), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f0eb103-4977-655c-8003-960bf33c5dea'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-01-06T14:58:48.283068+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f0eb103-4973-6279-8002-e1de8f6fc5b7'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'input': 'start', 'step1': 'done', 'step2': 'done'}, next=('step_3',), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f0eb103-4973-6279-8002-e1de8f6fc5b7'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-01-06T14:58:48.281356+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f0eb101-0d38-6168-8001-39b6477fe025'}}, tasks=(PregelTask(id='1290ba85-3dea-ad22-be70-1195efd97d1

### Time Travel

In [16]:
graph.get_state(
    {
        "configurable": {
            "thread_id": "thread-1",
            "checkpoint_id": "1f0eb0f4-77a8-61ac-8002-fd351e816472",
        }
    }
)

StateSnapshot(values={}, next=(), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_id': '1f0eb0f4-77a8-61ac-8002-fd351e816472'}}, metadata=None, created_at=None, parent_config=None, tasks=(), interrupts=())

In [17]:
graph.invoke(
    None,
    {
        "configurable": {
            "thread_id": "thread-1",
            "checkpoint_id": "1f0eb0f4-77a8-61ac-8002-fd351e816472",
        }
    },
)

EmptyInputError: Received no input for __start__

In [18]:
list(graph.get_state_history(config))

[StateSnapshot(values={'input': 'start', 'step1': 'done', 'step2': 'done'}, next=(), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f0eb103-4977-655c-8003-960bf33c5dea'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-01-06T14:58:48.283068+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f0eb103-4973-6279-8002-e1de8f6fc5b7'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'input': 'start', 'step1': 'done', 'step2': 'done'}, next=('step_3',), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f0eb103-4973-6279-8002-e1de8f6fc5b7'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-01-06T14:58:48.281356+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f0eb101-0d38-6168-8001-39b6477fe025'}}, tasks=(PregelTask(id='1290ba85-3dea-ad22-be70-1195efd97d1

In [19]:
graph.update_state({"configurable": {"thread_id": 'thread-1', 'checkpoint_id': "1f06f838-67e5-63a7-8000-c9b854c9e955", "checkpoint_ns": ""}}, {'topic':'samosa'})

{'configurable': {'thread_id': 'thread-1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f0eb106-12b4-6d28-8000-6bef65ef4b49'}}

In [20]:
list(graph.get_state_history(config))

[StateSnapshot(values={}, next=('step_1',), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f0eb106-12b4-6d28-8000-6bef65ef4b49'}}, metadata={'source': 'update', 'step': 0, 'parents': {}}, created_at='2026-01-06T15:00:03.071722+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f06f838-67e5-63a7-8000-c9b854c9e955'}}, tasks=(PregelTask(id='9256bac9-6dea-8aa2-7217-dfbf0a8bf6b6', name='step_1', path=('__pregel_pull', 'step_1'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'input': 'start', 'step1': 'done', 'step2': 'done'}, next=(), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f0eb103-4977-655c-8003-960bf33c5dea'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-01-06T14:58:48.283068+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoi

In [22]:
graph.invoke(
    None,
    {
        "configurable": {
            "thread_id": "thread-1",
            "checkpoint_id": "1f0eb106-12b4-6d28-8000-6bef65ef4b49",
        }
    },
)

✅ Step 1 executed


KeyError: 'input'

In [23]:
list(graph.get_state_history(config))

[StateSnapshot(values={}, next=('step_1',), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f0eb106-12b4-6d28-8000-6bef65ef4b49'}}, metadata={'source': 'update', 'step': 0, 'parents': {}}, created_at='2026-01-06T15:00:03.071722+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f06f838-67e5-63a7-8000-c9b854c9e955'}}, tasks=(PregelTask(id='9256bac9-6dea-8aa2-7217-dfbf0a8bf6b6', name='step_1', path=('__pregel_pull', 'step_1'), error="KeyError('input')", interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'input': 'start', 'step1': 'done', 'step2': 'done'}, next=(), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f0eb103-4977-655c-8003-960bf33c5dea'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-01-06T14:58:48.283068+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns'